# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd
import numpy as np

url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

def clean_cols(df):
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_", regex=False)
    )
    return df

df1, df2, df3 = clean_cols(df1), clean_cols(df2), clean_cols(df3)


In [2]:
all_cols = sorted(set(df1.columns) | set(df2.columns) | set(df3.columns))

df1 = df1.reindex(columns=all_cols)
df2 = df2.reindex(columns=all_cols)
df3 = df3.reindex(columns=all_cols)

df = pd.concat([df1, df2, df3], ignore_index=True)
df.head()

,customer,customer_lifetime_value,education,gender,income,monthly_premium_auto,number_of_open_complaints,policy_type,st,state,total_claim_amount,vehicle_class
0,RB50392,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Washington,NaN,2.704934,Four-Door Car
1,QZ44356,697953.59%,Bachelor,F,0.0,94.0,1/0/00,Personal Auto,Arizona,NaN,1131.464935,Four-Door Car
2,AI49188,1288743.17%,Bachelor,F,48767.0,108.0,1/0/00,Personal Auto,Nevada,NaN,566.472247,Two-Door Car
3,WW63253,764586.18%,Bachelor,M,0.0,106.0,1/0/00,Corporate Auto,California,NaN,529.881344,SUV
4,GA49547,536307.65%,High School or Below,M,36357.0,68.0,1/0/00,Personal Auto,Washington,NaN,17.269323,Four-Door Car


In [4]:
df = df.drop_duplicates()

obj_cols = df.select_dtypes(include="object").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())

df = df.replace(
    {"": np.nan, "NA": np.nan, "N/A": np.nan, "na": np.nan, "null": np.nan, "?": np.nan}
)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9135 entries, 0 to 12073
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9134 non-null   object 
 1   customer_lifetime_value    2057 non-null   object 
 2   education                  9134 non-null   object 
 3   gender                     9012 non-null   object 
 4   income                     9134 non-null   float64
 5   monthly_premium_auto       9134 non-null   float64
 6   number_of_open_complaints  2064 non-null   object 
 7   policy_type                9134 non-null   object 
 8   st                         2064 non-null   object 
 9   state                      7070 non-null   object 
 10  total_claim_amount         9134 non-null   float64
 11  vehicle_class              9134 non-null   object 
dtypes: float64(3), object(9)
memory usage: 927.8+ KB


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [8]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)
df = clean_cols(df)
df.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type', 'month'],
      dtype='object')

In [12]:
revenue_by_channel = (
    df.pivot_table(
        values=revenue_col,
        index="sales_channel",
        aggfunc="sum"
    )
    .round(2)
    .sort_values(by=revenue_col, ascending=False)
)

revenue_by_channel

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [11]:
clv_pivot = (
    df.pivot_table(
        values="customer_lifetime_value",
        index="sales_channel",
        columns="education",
        aggfunc="mean"
    )
    .round(2)
)

clv_pivot

education,Bachelor,College,Doctor,High School or Below,Master
sales_channel,,,,,
Agent,7818.19,7735.55,7466.31,8389.75,8752.90
Branch,7833.43,8115.45,6347.77,8490.61,7996.99
Call Center,7958.46,7726.10,8327.77,8505.79,8441.07
Web,7413.01,8070.94,7888.43,8226.10,6942.40


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here